In [1]:
# ============================================================
# RESOLVEAI
# 02 - TEXT PREPROCESSING
# ============================================================


# ============================================================
# CELL 1 — IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import re
import string

from pathlib import Path

import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

print("Libraries imported successfully.")


# ============================================================
# CELL 2 — DOWNLOAD NLTK RESOURCES
# ============================================================

# Run this once if the resources are not already downloaded.

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

print("NLTK resources ready.")


# ============================================================
# CELL 3 — LOAD RAW DATASET
# ============================================================

DATA_PATH = Path("../data/raw/tickets.csv")

df = pd.read_csv(DATA_PATH)

print("Raw dataset loaded.")
print(f"Shape: {df.shape}")


# ============================================================
# CELL 4 — KEEP ENGLISH TICKETS
# ============================================================

print("Language distribution before filtering:")
print(df["language"].value_counts())

english_df = df[
    df["language"].str.lower() == "en"
].copy()

print("\nEnglish dataset:")
print(f"Rows: {len(english_df):,}")
print(f"Columns: {english_df.shape[1]}")


# ============================================================
# CELL 5 — KEEP ONLY RELEVANT COLUMNS
# ============================================================

# We keep the original identifiers and useful labels.
#
# We will NOT use every column as a model feature.
# Metadata such as tags, answer, etc. will be investigated
# separately to avoid data leakage.

selected_columns = [
    "subject",
    "body",
    "type",
    "queue",
    "priority"
]

english_df = english_df[selected_columns].copy()

print("Selected columns:")
print(english_df.columns.tolist())


# ============================================================
# CELL 6 — CHECK TARGET MISSING VALUES
# ============================================================

print("\nMissing target values:")

print(
    english_df[
        ["queue", "priority", "type"]
    ].isna().sum()
)


# ============================================================
# CELL 7 — HANDLE MISSING SUBJECTS
# ============================================================

# Subject has missing values, but body does not.
#
# We do NOT delete these rows.
# A missing subject simply becomes an empty string.

english_df["subject"] = (
    english_df["subject"]
    .fillna("")
    .astype(str)
)

english_df["body"] = (
    english_df["body"]
    .fillna("")
    .astype(str)
)


# ============================================================
# CELL 8 — CREATE COMBINED RAW TEXT
# ============================================================

english_df["text_raw"] = (
    english_df["subject"].str.strip()
    + " "
    + english_df["body"].str.strip()
).str.strip()

# Remove accidental extra whitespace
english_df["text_raw"] = (
    english_df["text_raw"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print("Combined text created.")

display(
    english_df[
        ["subject", "body", "text_raw", "queue", "priority"]
    ].head()
)


# ============================================================
# CELL 9 — REMOVE EMPTY TEXT RECORDS
# ============================================================

empty_text = (
    english_df["text_raw"]
    .str.strip()
    .eq("")
)

print(f"Completely empty text records: {empty_text.sum()}")

english_df = english_df[
    ~empty_text
].copy()

print(
    f"Dataset after removing empty text: "
    f"{len(english_df):,}"
)


# ============================================================
# CELL 10 — TEXT CLEANING FUNCTION
# ============================================================

def clean_text(text):
    """
    Basic NLP text normalization.

    Steps:
    1. Lowercase
    2. Remove HTML tags
    3. Remove URLs
    4. Replace email addresses
    5. Remove punctuation
    6. Remove extra whitespace
    """

    text = str(text)

    # 1. Lowercase
    text = text.lower()

    # 2. Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # 3. Remove URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # 4. Remove email addresses
    text = re.sub(
        r"\b[\w\.-]+@[\w\.-]+\.\w+\b",
        " ",
        text
    )

    # 5. Remove punctuation
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    # 6. Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ============================================================
# CELL 11 — APPLY BASIC CLEANING
# ============================================================

english_df["text_basic_clean"] = (
    english_df["text_raw"]
    .apply(clean_text)
)

display(
    english_df[
        ["text_raw", "text_basic_clean"]
    ].head(10)
)


# ============================================================
# CELL 12 — TOKENIZATION
# ============================================================

english_df["tokens"] = (
    english_df["text_basic_clean"]
    .apply(word_tokenize)
)

display(
    english_df[
        ["text_basic_clean", "tokens"]
    ].head(5)
)


# ============================================================
# CELL 13 — STOPWORDS
# ============================================================

stop_words = set(
    stopwords.words("english")
)

print(f"Number of English stopwords: {len(stop_words)}")

print("\nSample stopwords:")
print(list(stop_words)[:30])


# ============================================================
# CELL 14 — STOPWORD REMOVAL
# ============================================================

def remove_stopwords(tokens):
    return [
        token
        for token in tokens
        if token not in stop_words
    ]


english_df["tokens_no_stopwords"] = (
    english_df["tokens"]
    .apply(remove_stopwords)
)

display(
    english_df[
        [
            "tokens",
            "tokens_no_stopwords"
        ]
    ].head(5)
)


# ============================================================
# CELL 15 — LEMMATIZATION
# ============================================================

lemmatizer = WordNetLemmatizer()


def lemmatize_tokens(tokens):
    return [
        lemmatizer.lemmatize(token)
        for token in tokens
    ]


english_df["tokens_lemma"] = (
    english_df["tokens_no_stopwords"]
    .apply(lemmatize_tokens)
)

display(
    english_df[
        [
            "tokens_no_stopwords",
            "tokens_lemma"
        ]
    ].head(5)
)


# ============================================================
# CELL 16 — CREATE FINAL CLEAN TEXT
# ============================================================

english_df["text_clean"] = (
    english_df["tokens_lemma"]
    .apply(lambda tokens: " ".join(tokens))
)

display(
    english_df[
        [
            "text_raw",
            "text_basic_clean",
            "text_clean"
        ]
    ].head(10)
)


# ============================================================
# CELL 17 — COMPARE TEXT LENGTH
# ============================================================

english_df["raw_word_count"] = (
    english_df["text_raw"]
    .str.split()
    .str.len()
)

english_df["clean_word_count"] = (
    english_df["text_clean"]
    .str.split()
    .str.len()
)

length_comparison = pd.DataFrame({
    "raw_word_count": english_df["raw_word_count"],
    "clean_word_count": english_df["clean_word_count"]
})

display(
    length_comparison.describe()
)


# ============================================================
# CELL 18 — BEFORE / AFTER EXAMPLES
# ============================================================

comparison = english_df[
    [
        "text_raw",
        "text_basic_clean",
        "text_clean"
    ]
].sample(
    10,
    random_state=42
)

display(comparison)


# ============================================================
# CELL 19 — CHECK EMPTY CLEAN TEXT
# ============================================================

empty_clean = (
    english_df["text_clean"]
    .str.strip()
    .eq("")
)

print(
    f"Rows with empty cleaned text: "
    f"{empty_clean.sum()}"
)


# ============================================================
# CELL 20 — CHECK DUPLICATES AFTER CLEANING
# ============================================================

print(
    "Duplicate raw text:",
    english_df["text_raw"].duplicated().sum()
)

print(
    "Duplicate cleaned text:",
    english_df["text_clean"].duplicated().sum()
)


# ============================================================
# CELL 21 — TARGET DISTRIBUTIONS AFTER PREPROCESSING
# ============================================================

print("\nQUEUE DISTRIBUTION")
print("=" * 60)

display(
    english_df["queue"]
    .value_counts()
    .to_frame("count")
)


print("\nPRIORITY DISTRIBUTION")
print("=" * 60)

display(
    english_df["priority"]
    .value_counts()
    .to_frame("count")
)


print("\nTYPE DISTRIBUTION")
print("=" * 60)

display(
    english_df["type"]
    .value_counts()
    .to_frame("count")
)


# ============================================================
# CELL 22 — CHECK CLASS IMBALANCE
# ============================================================

queue_balance = (
    english_df["queue"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("percentage")
)

priority_balance = (
    english_df["priority"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("percentage")
)

print("QUEUE BALANCE")
display(queue_balance)

print("\nPRIORITY BALANCE")
display(priority_balance)


# ============================================================
# CELL 23 — FINAL MODELING DATASET
# ============================================================

model_df = english_df[
    [
        "text_raw",
        "text_basic_clean",
        "text_clean",
        "type",
        "queue",
        "priority"
    ]
].copy()

print("Final modeling dataset shape:")
print(model_df.shape)

display(model_df.head())


# ============================================================
# CELL 24 — SAVE PROCESSED DATASET
# ============================================================

OUTPUT_PATH = Path(
    "../data/processed/english_tickets_processed.csv"
)

model_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    f"Processed dataset saved to:\n{OUTPUT_PATH}"
)


# ============================================================
# CELL 25 — FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("RESOLVEAI — PREPROCESSING SUMMARY")
print("=" * 70)

print(f"""
Original dataset              : 28,587 tickets
English tickets               : {len(model_df):,}

Final modeling records        : {len(model_df):,}

Missing subjects handled      : Yes
Missing bodies                : {model_df['text_raw'].eq('').sum()}

Text representations:
    1. text_raw
    2. text_basic_clean
    3. text_clean

Tokenization                  : Yes
Stopword removal              : Yes
Lemmatization                 : Yes

Queue classes                 : {model_df['queue'].nunique()}
Priority classes              : {model_df['priority'].nunique()}
Type classes                  : {model_df['type'].nunique()}
""")

print("=" * 70)
print("PREPROCESSING COMPLETE")
print("=" * 70)

Libraries imported successfully.


[nltk_data] Downloading package punkt to C:\Users\Shraddha
[nltk_data]     Sharma\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to C:\Users\Shraddha
[nltk_data]     Sharma\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Shraddha
[nltk_data]     Sharma\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Shraddha
[nltk_data]     Sharma\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Shraddha
[nltk_data]     Sharma\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


NLTK resources ready.
Raw dataset loaded.
Shape: (28587, 16)
Language distribution before filtering:
language
en    16338
de    12249
Name: count, dtype: int64

English dataset:
Rows: 16,338
Columns: 16
Selected columns:
['subject', 'body', 'type', 'queue', 'priority']

Missing target values:
queue       0
priority    0
type        0
dtype: int64
Combined text created.


,subject,body,text_raw,queue,priority
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Account Disruption Dear Customer Support Team,...",Technical Support,high
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Query About Smart Home System Integration Feat...,Returns and Exchanges,medium
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",Inquiry Regarding Invoice Details Dear Custome...,Billing and Payments,low
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Question About Marketing Agency Software Compa...,Sales and Pre-Sales,medium
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...","Feature Query Dear Customer Support,\n\nI hope...",Technical Support,high


Completely empty text records: 0
Dataset after removing empty text: 16,338


...


PRIORITY BALANCE


,percentage
priority,
medium,40.51
high,38.84
low,20.65


Final modeling dataset shape:
(16338, 6)


,text_raw,text_basic_clean,text_clean,type,queue,priority
1,"Account Disruption Dear Customer Support Team,...",account disruption dear customer support teamn...,account disruption dear customer support teamn...,Incident,Technical Support,high
2,Query About Smart Home System Integration Feat...,query about smart home system integration feat...,query smart home system integration feature de...,Request,Returns and Exchanges,medium
3,Inquiry Regarding Invoice Details Dear Custome...,inquiry regarding invoice details dear custome...,inquiry regarding invoice detail dear customer...,Request,Billing and Payments,low
4,Question About Marketing Agency Software Compa...,question about marketing agency software compa...,question marketing agency software compatibili...,Problem,Sales and Pre-Sales,medium
5,"Feature Query Dear Customer Support,\n\nI hope...",feature query dear customer supportnni hope th...,feature query dear customer supportnni hope me...,Request,Technical Support,high


Processed dataset saved to:
..\data\processed\english_tickets_processed.csv


RESOLVEAI — PREPROCESSING SUMMARY

Original dataset              : 28,587 tickets
English tickets               : 16,338

Final modeling records        : 16,338

Missing subjects handled      : Yes
Missing bodies                : 0

Text representations:
    1. text_raw
    2. text_basic_clean
    3. text_clean

Tokenization                  : Yes
Stopword removal              : Yes
Lemmatization                 : Yes

Queue classes                 : 10
Priority classes              : 3
Type classes                  : 4

PREPROCESSING COMPLETE
